### 개요
- 프로젝트 주제: RNN을 활용한 장르별 노래 가사 생성 시스템
- 데이터 출처: **멜론 사이트에서 장르별 50곡 가사 크롤링**
- 한국어 형태소 분석기 konlpy의 Okt 사용하여 토큰화
- LSTM 모델 적용

# 공통 실행 환경
Python 3.11 이상과 Java가 있는 환경에서 이 절을 한 번 실행한 후, 멜론 장르별 수집 또는 발라드 TSV 학습 예제를 선택하세요. 다른 형태소 분석기가 JVM을 먼저 시작했다면 새 커널에서 공통 실행 환경을 실행하세요.

In [ ]:
%pip install "requests>=2.34.2,<3" "beautifulsoup4>=4.15,<5" "tensorflow>=2.21,<3" "scikit-learn>=1.9.1,<2" "konlpy==0.6.0"

In [ ]:
import re
from pathlib import Path

import jpype
import konlpy
import requests
import tensorflow as tf
from bs4 import BeautifulSoup
from konlpy.tag import Okt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Embedding, LSTM

### 형태소 분석기
- morphs( text )
  - 텍스트에서 형태소를 반환한다
 
- nouns( text )
  - 텍스트에서 명사를 반환한다
 
- pos( text )
  - 텍스트에서 품사 정보를 부착하여 반환한다
- sentences( text )
  - 텍스트에서 문장별로 변환한다
- Hannanum: 한나눔. KAIST Semantic Web Research Center 개발.

  - http://semanticweb.kaist.ac.kr/hannanum/

- Kkma: 꼬꼬마. 서울대학교 IDS(Intelligent Data Systems) 연구실 개발.

  - http://kkma.snu.ac.kr/

- Komoran: 코모란. Shineware에서 개발.

  - https://github.com/shin285/KOMORAN

- Mecab: 메카브. 일본어용 형태소 분석기를 한국어를 사용할 수 있도록 수정.

  - https://bitbucket.org/eunjeon/mecab-ko

- Open Korean Text: 오픈 소스 한국어 분석기. 과거 트위터 형태소 분석기.

  - https://github.com/open-korean-text/open-korean-text

In [ ]:
# Okt가 사용하는 Java 클래스만 불러옵니다.
java_dir = Path(konlpy.__file__).parent / "java"
okt_libraries = ["bin", "scala-library-2.12.3.jar", "open-korean-text-2.1.0.jar", "twitter-text-1.14.7.jar"]
if not jpype.isJVMStarted():
    jpype.startJVM("-Dfile.encoding=UTF8", "-Xmx1024m",
                   classpath=[str(java_dir / name) for name in okt_libraries],
                   convertStrings=True)
okt = Okt()

In [ ]:
def clean_corpus(songs):
    valid = [song for song in songs if song.strip() and "[" not in song and "]" not in song]
    unique = list(dict.fromkeys(valid))
    print(f"원래 데이터 크기: {len(songs)}")
    print(f"공백과 대괄호가 있는 문장을 제외한 크기: {len(valid)}")
    print(f"중복 제거 후 크기: {len(unique)}")
    if len(unique) < 2:
        raise ValueError("학습용과 평가용으로 나눌 수 있는 가사가 두 곡 이상 필요합니다.")
    return unique


def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence.lower())
    return re.sub(r"[^가-힣0-9a-z?.!,¿]+", " ", sentence).strip()


def tokenize_songs(sentences):
    return [["<start>", *okt.morphs(sentence), "<end>"] for sentence in sentences]

###토큰화2. 단어장 만들기 (tokenizer)

In [ ]:
def tokenize(songs):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=15000, filters=" ", oov_token="<unk>")
    tokenizer.fit_on_texts(songs)
    sequences = tokenizer.texts_to_sequences(songs)
    tensor = tf.keras.utils.pad_sequences(sequences, padding="post", maxlen=15)
    return tensor, tokenizer

### LSTM 모델과 가사 생성

In [ ]:
class TextGenerator(tf.keras.Model):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = Embedding(vocab_size, embedding_size)
        self.rnn_1 = LSTM(hidden_size, return_sequences=True, return_state=True)
        self.rnn_2 = LSTM(hidden_size, return_sequences=True, return_state=True)
        self.linear = Dense(vocab_size)

    def recurrent(self, tokens, states=(None, None)):
        first, h1, c1 = self.rnn_1(self.embedding(tokens), initial_state=states[0])
        second, h2, c2 = self.rnn_2(first, initial_state=states[1])
        return second, ((h1, c1), (h2, c2))

    def call(self, tokens):
        hidden, _ = self.recurrent(tokens)
        return self.linear(hidden)

    def next_token(self, tokens, states):
        hidden, states = self.recurrent(tokens, states)
        logits = self.linear(hidden[:, -1, :])
        return int(tf.argmax(logits[0]).numpy()), states

In [ ]:
def generate_text(model, tokenizer, init_sentence="<start>", max_len=20):
    words = tokenizer.texts_to_sequences([init_sentence])[0]
    if not words:
        raise ValueError("시작 문장에 단어를 입력하세요.")
    end_token = tokenizer.word_index["<end>"]
    tokens = tf.constant([words])
    states = (None, None)
    while len(words) < max_len:
        word, states = model.next_token(tokens, states)
        words.append(word)
        if word == end_token:
            break
        tokens = tf.constant([[word]])
    return " ".join(tokenizer.index_word[word] for word in words if word)

# 멜론 장르별 TOP 50 크롤링 하기
  - 원하는 장르를 선택하고 TOP 50을 크롤링 합니다.


In [ ]:
genre_codes = {
    "발라드": "GN0100", "댄스": "GN0200", "힙합": "GN0300", "알앤비": "GN0400",
    "인디": "GN0500", "록": "GN0600", "트로트": "GN0700", "포크": "GN0800",
}

def collect_songs(session, genre_code):
    response = session.get("https://www.melon.com/genre/song_list.htm",
                           params={"gnrCode": genre_code}, timeout=(5, 15))
    response.raise_for_status()
    chart = BeautifulSoup(response.content, "html.parser")
    song_ids = {}
    for link in chart.select("a.song_info[href]"):
        match = re.search(r"goSongDetail\(\s*['\"]?(\d+)['\"]?\s*\)", link["href"])
        if match:
            song_ids[match.group(1)] = None
        if len(song_ids) == 50:
            break
    if not song_ids:
        raise ValueError("장르 목록에서 곡 번호를 찾지 못했습니다.")

    songs = []
    for song_id in song_ids:
        detail = session.get("https://www.melon.com/song/detail.htm",
                             params={"songId": song_id}, timeout=(5, 15))
        detail.raise_for_status()
        lyric = BeautifulSoup(detail.content, "html.parser").select_one("div.lyric")
        if lyric:
            text = lyric.get_text(" ", strip=True)
            if text:
                songs.append(text)
    return songs


while True:
    genre = input("음악 장르 입력(" + ", ".join(genre_codes) + "): ").strip()
    if genre in genre_codes:
        break
    print("장르 입력이 잘못되었습니다. 다시 입력해 주십시오.")

with requests.Session() as session:
    session.headers["User-Agent"] = "Mozilla/5.0"
    Song = collect_songs(session, genre_codes[genre])

In [ ]:
len(Song)

In [ ]:
corpus = clean_corpus(Song)

In [ ]:
corpus_tokenized = [preprocess_sentence(sentence) for sentence in corpus]
corpus_tokenized[:10]

In [ ]:
song_list = tokenize_songs(corpus_tokenized)

In [ ]:
tensor, tokenizer = tokenize(song_list)

In [ ]:
# 단어장의 1~10번째 단어 확인해보기

for idx in tokenizer.index_word:
    print(idx, ":", tokenizer.index_word[idx])

    if idx >= 10: break

In [ ]:
src_input = tensor[:, :-1] # 소스문장 생성 (마지막 단어 1개 빼고 다 가져오기)

tgt_input = tensor[:, 1:]  # 타켓문장 생성 (첫번째 단어 1개 빼고 다 가져오기)

print(src_input[0])
print(tgt_input[0])

In [ ]:
# 소스 문장과 타깃 문장의 단어를 확인합니다. 0은 패딩입니다.
for sequence in (src_input[0], tgt_input[0]):
    print(" ".join(tokenizer.index_word[idx] for idx in sequence if idx))

###평가 데이터셋 분리

In [ ]:
enc_train, enc_val, dec_train, dec_val = train_test_split(
    src_input, tgt_input, test_size=0.2, shuffle=True, random_state=121
)

In [ ]:
print("Source Train:", enc_train.shape)
print("Target Train:", dec_train.shape)

In [ ]:
# 실제 단어장과 <unk>, 패딩에 필요한 출력만 계산합니다.
vocab_size = min(tokenizer.num_words, len(tokenizer.word_index) + 1)
model = TextGenerator(vocab_size, embedding_size=700, hidden_size=350)

In [ ]:
optimizer = tf.keras.optimizers.Adam()
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
model.compile(loss=loss, optimizer=optimizer)
model.fit(enc_train, dec_train, epochs=200, batch_size=32)

In [ ]:
# 크롤링한 댄스 50곡으로 200번 학습한 결과, okt 사용, <start>, <end> 추가 

print('예시1.', generate_text(model, tokenizer, init_sentence='힘내', max_len=50))
print('예시2.', generate_text(model, tokenizer, init_sentence='슬픔', max_len=50))
print('예시3.', generate_text(model, tokenizer, init_sentence='비가 내려', max_len=50))
print('예시4.', generate_text(model, tokenizer, init_sentence='i love', max_len=50))

**이전 학습 실행에서 저장한 생성 예시**

> 예시1. <unk> 도 crazy 도 와 i i hmm hmm < < < > <end> 
> 예시2. <unk> 도 crazy 도 와 i i hmm hmm < < < > <end> 
> 예시3. <unk> 내려 을 도 와 i hmm hmm hmm < < end <end> 
> 예시4. i love 도 와 i i i hmm hmm < < < > <end>

# 모델링 팀의 성공적인 학습방법

### 개요
- 프로젝트 주제: RNN을 활용한 장르별 노래 가사 생성 시스템
- 데이터 출처: https://github.com/sohyeon98720/NLP/tree/master/KoGPT2-finetuning_lyrics/dataset
- 2011~2019년 발라드 곡 474개를 학습데이터로 활용함
- 한국어 형태소 분석기 konlpy의 Okt 사용하여 토큰화
- LSTM 모델 적용

In [ ]:
with open("/content/2011-2019_발라드.tsv", encoding="utf-8") as lyrics_file:
    raw_corpus = lyrics_file.read().splitlines()

print("데이터 크기:", len(raw_corpus))
print("Examples:", raw_corpus[:10])

In [ ]:
# 마지막곡 가사 출력
raw_corpus[-1]

In [ ]:
corpus = clean_corpus(raw_corpus)

In [ ]:
corpus_tokenized = [preprocess_sentence(sentence) for sentence in corpus]
corpus_tokenized[:10]

In [ ]:
song_list = tokenize_songs(corpus_tokenized)

In [ ]:
print("총 곡의 수:", len(song_list))
print("<start>의 개수:", sum(song.count("<start>") for song in song_list))
print("<end>의 개수:", sum(song.count("<end>") for song in song_list))

In [ ]:
tensor, tokenizer = tokenize(song_list)

In [ ]:
# 단어장의 1~10번째 단어 확인해보기

for idx in tokenizer.index_word:
    print(idx, ":", tokenizer.index_word[idx])

    if idx >= 10: break

In [ ]:
src_input = tensor[:, :-1] # 소스문장 생성 (마지막 단어 1개 빼고 다 가져오기)

tgt_input = tensor[:, 1:]  # 타켓문장 생성 (첫번째 단어 1개 빼고 다 가져오기)

print(src_input[0])
print(tgt_input[0])

In [ ]:
# 소스 문장과 타깃 문장의 단어를 확인합니다. 0은 패딩입니다.
for sequence in (src_input[0], tgt_input[0]):
    print(" ".join(tokenizer.index_word[idx] for idx in sequence if idx))

###평가 데이터셋 분리

In [ ]:
enc_train, enc_val, dec_train, dec_val = train_test_split(
    src_input, tgt_input, test_size=0.2, shuffle=True, random_state=121
)

In [ ]:
print("Source Train:", enc_train.shape)
print("Target Train:", dec_train.shape)

In [ ]:
# 실제 단어장과 <unk>, 패딩에 필요한 출력만 계산합니다.
vocab_size = min(tokenizer.num_words, len(tokenizer.word_index) + 1)
model = TextGenerator(vocab_size, embedding_size=700, hidden_size=350)

In [ ]:
optimizer = tf.keras.optimizers.Adam()
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
model.compile(loss=loss, optimizer=optimizer)
model.fit(enc_train, dec_train, epochs=200, batch_size=32)

In [ ]:
# 발라드 474곡으로 200번 학습한 결과, okt 사용, <start>, <end> 추가 

print('예시1.', generate_text(model, tokenizer, init_sentence='힘내', max_len=50))
print('예시2.', generate_text(model, tokenizer, init_sentence='슬픔', max_len=50))
print('예시3.', generate_text(model, tokenizer, init_sentence='i love', max_len=50))
print('예시3.', generate_text(model, tokenizer, init_sentence='비가 내려', max_len=50))

**이전 학습 실행에서 저장한 생성 예시**

> 예시1. 힘내 사랑 에 젠 하는 더 건 더 더 i 까지 you <end> 
> 예시2. 슬픔 상처 에 지우죠 그 널 사랑 한다 더 까지 <end> 
> 예시3. i love and i love time believe 뭘 over you feel first kiss <end> 
> 예시3. <unk> 내려 a 진작 를 그렇게 보고싶다 말 해줬으면 이렇게 끝나진 않을 걸 이 젠 늦었어 <end>